In [52]:
import pandas as pd 
import os
from tqdm import tqdm

In [3]:
f_ocupaciones = '../data/extracted_data/coe1/csv/conjunto_de_datos_coe1_enoe_2005_1t.csv'
ocupaciones = pd.read_csv(f_ocupaciones,usecols=[0,2,3,4,5,6,7,8,9,10,11,12,59],na_values=['',' ','  '])

In [4]:
f_sexo = '../data/extracted_data/sdem/csv/conjunto_de_datos_sdem_enoe_2005_1t.csv'
sexo = pd.read_csv(f_sexo,usecols=[0,8,9,10,11,12,13,14,15,16,17,18,21],na_values=['',' ','  '])

In [5]:
columnas_llave =['r_def','ent','con','upm','d_sem','n_pro_viv','v_sel','n_hog','h_mud','n_ent','per','n_ren']
ocupacion_por_sexo=pd.merge(ocupaciones,sexo,on=columnas_llave,how='left')

In [6]:
ocupacion_sexo_limpio=ocupacion_por_sexo.dropna(subset=['p3','sex'])

In [7]:
prefijos_validos=('13','16','22','26')
f_ocup_sex_stem=ocupacion_sexo_limpio[ocupacion_sexo_limpio['p3'].astype(str).str.startswith(prefijos_validos)]

In [8]:
ocup_sex_stem=f_ocup_sex_stem[['p3','sex']]
ocup_sex_stem.rename(columns={'p3':'ocupación','sex':'sexo'},inplace=True)

In [9]:
ocup_sex_stem.to_csv('../data/merged/ocup_sex_2005_1t.csv')

In [77]:
ruta_raiz = 'D:/HackODS/data/extracted_data/'

ID_ruta_coe1 = {}
ID_ruta_sdem = {}

columnas_llave =['ent', 'con', 'v_sel', 'n_hog', 'h_mud', 'n_pro_viv', 'per', 'n_ren']
columnas_coe1 = columnas_llave + ['p3']
columnas_sdem = columnas_llave + ['sex']

prefijos_validos=('13','16','22','26')

for root, dirs, files in os.walk(ruta_raiz):
    for nombre in files:
        nombre_lower = nombre.lower()
        id_archivo = nombre_lower.replace('coe1', '').replace('sdem', '').replace('.csv', '').strip('_')
        
        ruta_completa = os.path.join(root, nombre)
        
        if 'coe1' in nombre_lower and nombre.endswith('.csv'):
            ID_ruta_coe1[id_archivo] = ruta_completa
        elif 'sdem' in nombre_lower and nombre.endswith('.csv'):
            ID_ruta_sdem[id_archivo] = ruta_completa
            
lista_resultados = []

parejas = [ID for ID in ID_ruta_coe1.items() if ID[0] in ID_ruta_sdem]

for ID, ruta_c in tqdm(parejas):
    try:
        ocupaciones = pd.read_csv(ruta_c, dtype=str, encoding='latin-1', low_memory=False)
        sexo = pd.read_csv(ID_ruta_sdem[ID], dtype=str, encoding='latin-1', low_memory=False)

        ocupaciones.columns = ocupaciones.columns.str.lower().str.strip()
        sexo.columns = sexo.columns.str.lower().str.strip()

        cols_coe1 = [c for c in columnas_coe1 if c in ocupaciones.columns]
        cols_sdem = [c for c in columnas_sdem if c in sexo.columns]      

        ocupaciones_columnas = ocupaciones[columnas_coe1]
        sexo_columnas = sexo[columnas_sdem]
            
        ocup_sexo = pd.merge(ocupaciones,sexo, on=columnas_llave, how='inner')

        if not ocup_sexo.empty:
            ocup_sexo['p3'] = ocup_sexo['p3'].str.strip()

            mask = ocup_sexo['p3'].str.startswith(prefijos_validos, na=False)
            ocup_sexo_final=ocup_sexo[mask].copy()

            if not ocup_sexo_final.empty:
                ocup_sexo_final['periodo'] = ID
                lista_resultados.append(ocup_sexo_final[['ent','p3','sex','periodo']])
    
    except Exception as e:
        print(f" Error al procesar el par {ID}: {e}")

if lista_resultados:
    ocup_sexo_final_csv = pd.concat(lista_resultados, ignore_index=True)
    print(f" Por fin quedó :') Registros totales: {len(ocup_sexo_final)}")
    ocup_sexo_final_csv.to_csv('../data/extracted_data/datos_finales_stem.csv', index=False)

 99%|█████████████████████████████████████▌| 82/83 [14:16<00:09,  9.79s/it]

 Error al procesar el par conjunto_de_datos__enoe_2025_3t: "['ent'] not in index"


100%|██████████████████████████████████████| 83/83 [14:27<00:00, 10.46s/it]

 Error al procesar el par conjunto_de_datos__enoe_2025_4t: "['ent'] not in index"
 Por fin quedó :') Registros totales: 13255


In [80]:
ocup_sexo_final_csv

,ent,p3,sex,periodo
0,9,2651,1,conjunto_de_datos__enoen_2020_3t
1,9,2644,1,conjunto_de_datos__enoen_2020_3t
2,9,2644,1,conjunto_de_datos__enoen_2020_3t
3,9,2651,1,conjunto_de_datos__enoen_2020_3t
4,9,2644,1,conjunto_de_datos__enoen_2020_3t
...,...,...,...,...
834062,22,2252,1,conjunto_de_datos__enoe_2025_2t
834063,26,1311,2,conjunto_de_datos__enoe_2025_2t
834064,27,2643,1,conjunto_de_datos__enoe_2025_2t
834065,27,2271,1,conjunto_de_datos__enoe_2025_2t
